## Problem Statement: Build a mini-Transformer based next-word predictor from scratch

In [1]:
# Import the libraries and load the dataset
import re
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter

data = pd.read_csv('medium_data.csv')
data.head()

,id,url,title,subtitle,image,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/a-beginners-gui...,A Beginner’s Guide to Word Embedding with Gens...,NaN,1.png,850,8,8,Towards Data Science,2019-05-30
1,2,https://towardsdatascience.com/hands-on-graph-...,Hands-on Graph Neural Networks with PyTorch & ...,NaN,2.png,1100,11,9,Towards Data Science,2019-05-30
2,3,https://towardsdatascience.com/how-to-use-ggpl...,How to Use ggplot2 in Python,A Grammar of Graphics for Python,3.png,767,1,5,Towards Data Science,2019-05-30
3,4,https://towardsdatascience.com/databricks-how-...,Databricks: How to Save Files in CSV on Your L...,When I work on Python projects dealing…,4.jpeg,354,0,4,Towards Data Science,2019-05-30
4,5,https://towardsdatascience.com/a-step-by-step-...,A Step-by-Step Implementation of Gradient Desc...,One example of building neural…,5.jpeg,211,3,4,Towards Data Science,2019-05-30


In [2]:
# Specifying the Target Column
TEXT_COLUMN = "title"

# Cleaning and Converting the Data
texts = (
    data[TEXT_COLUMN]
    .dropna()
    .astype(str)
    .tolist()
)
print(texts)

['A Beginner’s Guide to Word Embedding with Gensim Word2Vec\xa0Model', 'Hands-on Graph Neural Networks with PyTorch & PyTorch Geometric', 'How to Use ggplot2 in\xa0Python', 'Databricks: How to Save Files in CSV on Your Local\xa0Computer', 'A Step-by-Step Implementation of Gradient Descent and Backpropagation', 'An Easy Introduction to SQL for Data Scientists', 'Hypothesis testing visualized', 'Introduction to Latent Matrix Factorization Recommender Systems', 'Which 2020 Candidate is the Best at\xa0Twitter?', 'What if AI model understanding were\xa0easy?', '<em class="markup--em markup--h3-em">What I Learned from (Two-time) Kaggle Grandmaster Abhishek\xa0Thakur</em>', 'Making a DotA2 Bot Using\xa0ML', 'Building A ‘Serverless’ Chrome Extension', 'How to Teach\xa0Code', 'Reinventing Personalization For Customer Experience', 'How to Automate Hyperparameter Optimization', 'Ideas: Design Methodologies for Data\xa0Sprints', 'RoboSomm Chapter 3: Wine Embeddings and a Wine Recommender', 'Data S

In [3]:
#  Limiting the Dataset Size
texts = texts[:5000]

# Initializing the Word Container and regular expression Extraction
words_in_data = []
for sentence in texts:
  words_in_data.extend(
      re.findall(r"\b[a-zA-Z]+\b",sentence.lower())
  )
print(words_in_data)

['a', 'beginner', 's', 'guide', 'to', 'word', 'embedding', 'with', 'gensim', 'model', 'hands', 'on', 'graph', 'neural', 'networks', 'with', 'pytorch', 'pytorch', 'geometric', 'how', 'to', 'use', 'in', 'python', 'databricks', 'how', 'to', 'save', 'files', 'in', 'csv', 'on', 'your', 'local', 'computer', 'a', 'step', 'by', 'step', 'implementation', 'of', 'gradient', 'descent', 'and', 'backpropagation', 'an', 'easy', 'introduction', 'to', 'sql', 'for', 'data', 'scientists', 'hypothesis', 'testing', 'visualized', 'introduction', 'to', 'latent', 'matrix', 'factorization', 'recommender', 'systems', 'which', 'candidate', 'is', 'the', 'best', 'at', 'twitter', 'what', 'if', 'ai', 'model', 'understanding', 'were', 'easy', 'em', 'class', 'markup', 'em', 'markup', 'em', 'what', 'i', 'learned', 'from', 'two', 'time', 'kaggle', 'grandmaster', 'abhishek', 'thakur', 'em', 'making', 'a', 'bot', 'using', 'ml', 'building', 'a', 'serverless', 'chrome', 'extension', 'how', 'to', 'teach', 'code', 'reinventin

In [4]:
# Counting Word Frequencies
counts = Counter(words_in_data)
print(counts)

Counter({'the': 1428, 'to': 1413, 'strong': 1049, 'a': 1046, 'of': 775, 'how': 705, 'and': 690, 'in': 634, 'your': 621, 'you': 579, 'markup': 552, 'for': 539, 'with': 404, 'is': 384, 's': 361, 'data': 309, 'why': 287, 'i': 286, 'class': 283, 'what': 276, 'on': 227, 'learning': 219, 'from': 200, 'an': 191, 'it': 188, 't': 183, 'be': 175, 'can': 172, 'are': 162, 'writing': 156, 'my': 153, 'design': 148, 'using': 138, 'about': 138, 'machine': 135, 'ux': 132, 'do': 124, 'not': 122, 'python': 121, 'that': 120, 'we': 116, 'science': 110, 'ai': 109, 'life': 109, 'should': 104, 'when': 103, 'need': 94, 'make': 93, 'time': 91, 'more': 91, 'as': 90, 'part': 85, 'business': 84, 'new': 84, 'at': 83, 'or': 82, 'have': 78, 'don': 77, 'get': 76, 'work': 74, 'up': 73, 'will': 72, 'by': 71, 'ways': 71, 'write': 70, 'guide': 69, 'use': 65, 'marketing': 65, 'first': 64, 'know': 63, 'good': 63, 'best': 62, 'deep': 62, 'this': 62, 'analysis': 60, 'better': 59, 'one': 58, 'things': 57, 'if': 56, 'em': 56, '

In [5]:
# Create the vocabulary list
vocab = ["<UNK>"] + [word for word, count in counts.most_common(2999)]
print(vocab)

['<UNK>', 'the', 'to', 'strong', 'a', 'of', 'how', 'and', 'in', 'your', 'you', 'markup', 'for', 'with', 'is', 's', 'data', 'why', 'i', 'class', 'what', 'on', 'learning', 'from', 'an', 'it', 't', 'be', 'can', 'are', 'writing', 'my', 'design', 'using', 'about', 'machine', 'ux', 'do', 'not', 'python', 'that', 'we', 'science', 'ai', 'life', 'should', 'when', 'need', 'make', 'time', 'more', 'as', 'part', 'business', 'new', 'at', 'or', 'have', 'don', 'get', 'work', 'up', 'will', 'by', 'ways', 'write', 'guide', 'use', 'marketing', 'first', 'know', 'good', 'best', 'deep', 'this', 'analysis', 'better', 'one', 'things', 'if', 'em', 're', 'out', 'writer', 'into', 'me', 'product', 'all', 'like', 'model', 'neural', 'case', 'stop', 'future', 'self', 'top', 'job', 'learned', 'intelligence', 'making', 'big', 'artificial', 'vs', 'building', 'tips', 'here', 'people', 'most', 'create', 'being', 'learn', 'build', 'google', 'networks', 'introduction', 'market', 'want', 'models', 'startup', 'our', 'so', 'so

In [6]:
# Creating the Word-to-ID Dictionary
word_to_id = {word: i for i, word in enumerate(vocab)}
print(word_to_id)

{'<UNK>': 0, 'the': 1, 'to': 2, 'strong': 3, 'a': 4, 'of': 5, 'how': 6, 'and': 7, 'in': 8, 'your': 9, 'you': 10, 'markup': 11, 'for': 12, 'with': 13, 'is': 14, 's': 15, 'data': 16, 'why': 17, 'i': 18, 'class': 19, 'what': 20, 'on': 21, 'learning': 22, 'from': 23, 'an': 24, 'it': 25, 't': 26, 'be': 27, 'can': 28, 'are': 29, 'writing': 30, 'my': 31, 'design': 32, 'using': 33, 'about': 34, 'machine': 35, 'ux': 36, 'do': 37, 'not': 38, 'python': 39, 'that': 40, 'we': 41, 'science': 42, 'ai': 43, 'life': 44, 'should': 45, 'when': 46, 'need': 47, 'make': 48, 'time': 49, 'more': 50, 'as': 51, 'part': 52, 'business': 53, 'new': 54, 'at': 55, 'or': 56, 'have': 57, 'don': 58, 'get': 59, 'work': 60, 'up': 61, 'will': 62, 'by': 63, 'ways': 64, 'write': 65, 'guide': 66, 'use': 67, 'marketing': 68, 'first': 69, 'know': 70, 'good': 71, 'best': 72, 'deep': 73, 'this': 74, 'analysis': 75, 'better': 76, 'one': 77, 'things': 78, 'if': 79, 'em': 80, 're': 81, 'out': 82, 'writer': 83, 'into': 84, 'me': 85,

In [7]:
# Creating the ID-to-Word Dictionary
id_to_word = {i: word for i, word in enumerate(vocab)}
print(id_to_word)

{0: '<UNK>', 1: 'the', 2: 'to', 3: 'strong', 4: 'a', 5: 'of', 6: 'how', 7: 'and', 8: 'in', 9: 'your', 10: 'you', 11: 'markup', 12: 'for', 13: 'with', 14: 'is', 15: 's', 16: 'data', 17: 'why', 18: 'i', 19: 'class', 20: 'what', 21: 'on', 22: 'learning', 23: 'from', 24: 'an', 25: 'it', 26: 't', 27: 'be', 28: 'can', 29: 'are', 30: 'writing', 31: 'my', 32: 'design', 33: 'using', 34: 'about', 35: 'machine', 36: 'ux', 37: 'do', 38: 'not', 39: 'python', 40: 'that', 41: 'we', 42: 'science', 43: 'ai', 44: 'life', 45: 'should', 46: 'when', 47: 'need', 48: 'make', 49: 'time', 50: 'more', 51: 'as', 52: 'part', 53: 'business', 54: 'new', 55: 'at', 56: 'or', 57: 'have', 58: 'don', 59: 'get', 60: 'work', 61: 'up', 62: 'will', 63: 'by', 64: 'ways', 65: 'write', 66: 'guide', 67: 'use', 68: 'marketing', 69: 'first', 70: 'know', 71: 'good', 72: 'best', 73: 'deep', 74: 'this', 75: 'analysis', 76: 'better', 77: 'one', 78: 'things', 79: 'if', 80: 'em', 81: 're', 82: 'out', 83: 'writer', 84: 'into', 85: 'me',

In [8]:
# Tokenization and ID Conversion
'''X, Y = [], []
for sentence in texts:
    tokens = re.findall(r"\b[a-zA-Z]+\b", sentence.lower())
    print(tokens)
    ids = [word_to_id.get(word, 0) for word in tokens]
    print(ids)
    
    # Creating Bigram Pairs (Input & Target)
    for i in range(len(ids) - 1):
        if ids[i] != 0 and ids[i + 1] != 0:
            X.append(ids[i])
            Y.append(ids[i + 1])
print(X)
print(Y)
'''

['a', 'beginner', 's', 'guide', 'to', 'word', 'embedding', 'with', 'gensim', 'model']
[4, 463, 15, 66, 2, 430, 1483, 13, 2002, 89]
['hands', 'on', 'graph', 'neural', 'networks', 'with', 'pytorch', 'pytorch', 'geometric']
[1484, 21, 552, 90, 113, 13, 367, 367, 1485]
['how', 'to', 'use', 'in', 'python']
[6, 2, 67, 8, 39]
['databricks', 'how', 'to', 'save', 'files', 'in', 'csv', 'on', 'your', 'local', 'computer']
[2999, 6, 2, 270, 970, 8, 2003, 21, 9, 707, 209]
['a', 'step', 'by', 'step', 'implementation', 'of', 'gradient', 'descent', 'and', 'backpropagation']
[4, 221, 63, 221, 399, 5, 0, 1486, 7, 1183]
['an', 'easy', 'introduction', 'to', 'sql', 'for', 'data', 'scientists']
[24, 203, 114, 2, 368, 12, 16, 271]
['hypothesis', 'testing', 'visualized']
[1487, 259, 0]
['introduction', 'to', 'latent', 'matrix', 'factorization', 'recommender', 'systems']
[114, 2, 0, 553, 2004, 554, 243]
['which', 'candidate', 'is', 'the', 'best', 'at', 'twitter']
[369, 1488, 14, 1, 72, 55, 322]
['what', 'if', '

In [25]:
CTX = 6  # number of previous words the model sees

X, Y = [], []
for sentence in texts:
    tokens = re.findall(r"\b[a-zA-Z]+\b", sentence.lower())
    ids = [word_to_id.get(word, 0) for word in tokens]

    for i in range(1, len(ids)):
        if ids[i] == 0:
            continue
        ctx = ids[max(0, i - CTX):i]
        ctx = [0] * (CTX - len(ctx)) + ctx   # left-pad to fixed length
        X.append(ctx)
        Y.append(ids[i])

In [26]:
# Converting to PyTorch Tensors
X = torch.tensor(X[:20000], dtype=torch.long)
Y = torch.tensor(Y[:20000], dtype=torch.long)
print(X)
print(Y)

tensor([[  0,   0,   0,   0,   0,   4],
        [  0,   0,   0,   0,   4, 463],
        [  0,   0,   0,   4, 463,  15],
        ...,
        [  1,   0, 348,   2,  37,   8],
        [  0, 348,   2,  37,   8,  44],
        [348,   2,  37,   8,  44,   7]])
tensor([ 463,   15,   66,  ...,   44,    7, 2661])


In [27]:
# Diagnostic print statements
print("Vocabulary size:", len(vocab))
print("Training examples:", len(X))

Vocabulary size: 3000
Training examples: 20000


In [13]:
#  The Model Container
'''model = nn.Sequential(
        nn.Embedding(len(vocab),32), # Embedding Layer
        nn.Linear(32, len(vocab))  # Linear (Output) Layer
)
print(model)
'''

Sequential(
  (0): Embedding(3000, 32)
  (1): Linear(in_features=32, out_features=3000, bias=True)
)


In [28]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d=64, ctx=CTX, heads=4, layers=2):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(ctx, d)          # positional encoding
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d,
                                           dropout=0.1, batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(d, vocab_size)

    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.size(1)))
        h = self.blocks(h)
        return self.out(h[:, -1])                # last position predicts next word

model = TinyTransformer(len(vocab))

In [29]:
# Initialize the Adam optimizer to update model weights based on language patterns.
# - model.parameters(): Links all trainable weights/biases of your next-word prediction model.
# - lr=0.01: Sets the step size (learning rate) to 0.01 for weight adjustments.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Define the loss function to evaluate how well the model predicts the correct next word.
# CrossEntropyLoss is ideal here because next-word prediction is a multi-class classification problem 
# where each unique word in your vocabulary represents an individual class.
loss_function = nn.CrossEntropyLoss()

In [32]:
# Loop through the entire dataset 20 times (20 epochs)
for epoch in range(20):
    
    # Put the model in training mode (activates dropout/batch normalization if present)
    model.train()
    
    # Forward Pass: Feed input text indices (X) to the model to get raw word predictions (logits)
    predictions = model(X)

    # Calculate Error: Compare predicted word distributions against the actual next-word targets (Y)
    loss = loss_function(predictions, Y)

    # Clear Gradients: Reset old gradients to zero so they don't accumulate from the previous epoch
    optimizer.zero_grad()

    # Backward Pass: Calculate the gradients (derivatives) of the loss with respect to model parameters
    loss.backward()

    # Update Weights: The Adam optimizer adjusts the network's weights based on the calculated gradients
    optimizer.step()
    if(epoch +  1) % 5 == 0:
        print(f"Epoch: {epoch + 1}, Loss: { loss.item():.4f}")

print("Training complete")

Epoch: 5, Loss: 6.5383
Epoch: 10, Loss: 5.9149
Epoch: 15, Loss: 5.3831
Epoch: 20, Loss: 4.8362
Training complete


In [20]:
# Predict the next word
'''def predict_next_word(word):
    word = word.lower()

    if word not in word_to_id or word_to_id[word] == 0:
        print("word not found in vocabulary")
        return

    model.eval()

    with torch.no_grad():
            input_id = torch.tensor([word_to_id[word]])
            scores = model(input_id)
            predicted_id = scores.argmax(dim=1).item()
    print("Input:",word)
    print("Predicted next word:" , id_to_word[predicted_id])
'''

In [21]:
predict_next_word("machine")

Input: machine
Predicted next word: learning


In [22]:
predict_next_word("learning")

Input: learning
Predicted next word: and


In [23]:
predict_next_word("and")

Input: and
Predicted next word: how


In [33]:
def predict_next_word(text):
    tokens = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    ids = [word_to_id.get(w, 0) for w in tokens][-CTX:]
    ids = [0] * (CTX - len(ids)) + ids
    model.eval()
    with torch.no_grad():
        scores = model(torch.tensor([ids]))
        predicted_id = scores.argmax(dim=1).item()
    print("Input:", text)
    print("Predicted next word:", id_to_word[predicted_id])

predict_next_word("introduction to machine")

Input: introduction to machine
Predicted next word: learning
